# DuckDB 基礎学習

DuckDBの基本的な動作と特徴を、実際に動かしながら理解しましょう。

## 目次
1. DuckDB のインポートと接続
2. **DuckDB の最大の特徴**：CSV をそのまま SQL でクエリ
3. インメモリDB vs ファイルDB（実体確認）
4. テーブル作成・データ投入・SELECT
5. 集計クエリ
6. JOIN クエリ

---
## 1. DuckDB のインポートと接続

`duckdb.connect()` でDB接続を作ります。  
引数なし → **インメモリDB**（プロセス終了でデータ消滅）  
引数あり → **ファイルDB**（`.duckdb` ファイルに永続化）

In [1]:
import duckdb
import os

# バージョン確認
print(f"DuckDB version: {duckdb.__version__}")

DuckDB version: 1.5.2


In [2]:
# インメモリDB（引数なし）
con = duckdb.connect()
print("インメモリDBに接続しました")

インメモリDBに接続しました


---
## 2. CSV をそのまま SQL でクエリ（DuckDBの最大の特徴）

DuckDB は **CSVファイルをテーブルとして直接クエリできます**。  
データベースにインポートする手間が不要！

In [ ]:
# CSVファイルのパス（このノートブックから見た相対パス）
DATA_DIR = "../data"

# CSVを直接SELECTする（fetchall() でリスト形式で取得）
rows = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{DATA_DIR}/customers.csv')
""").fetchall()

print("=== customers.csv の内容 ===")
for row in rows:
    print(row)

In [ ]:
rows = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{DATA_DIR}/products.csv')
""").fetchall()

print("=== products.csv の内容 ===")
for row in rows:
    print(row)

In [ ]:
rows = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{DATA_DIR}/sales.csv')
    LIMIT 5
""").fetchall()

print("=== sales.csv の最初の5件 ===")
for row in rows:
    print(row)

---
## 3. インメモリDB vs ファイルDB

DuckDB のDBは `.duckdb` という**1ファイル**が実体です。

In [6]:
# ファイルDBを作成
DB_PATH = "../mydb.duckdb"

file_con = duckdb.connect(DB_PATH)
print(f"ファイルDBを作成しました: {DB_PATH}")

# ファイルが実際に存在するか確認
if os.path.exists(DB_PATH):
    size_kb = os.path.getsize(DB_PATH) / 1024
    print(f"ファイルサイズ: {size_kb:.1f} KB")

ファイルDBを作成しました: ../mydb.duckdb
ファイルサイズ: 12.0 KB


In [ ]:
# ファイルDBにテーブルを作成してみる
file_con.execute("""
    CREATE TABLE IF NOT EXISTS test_table (id INTEGER, value TEXT)
""")
file_con.execute("INSERT INTO test_table VALUES (1, 'hello'), (2, 'world')")

# データ確認後、接続を閉じる
rows = file_con.execute("SELECT * FROM test_table").fetchall()
print("=== test_table ===")
for row in rows:
    print(row)
file_con.close()

# 再接続してもデータが残っていることを確認
file_con2 = duckdb.connect(DB_PATH)
print("\n再接続後も残っているか？")
rows2 = file_con2.execute("SELECT * FROM test_table").fetchall()
for row in rows2:
    print(row)
file_con2.close()

---
## 4. テーブル作成・データ投入・SELECT

CSVを読み込んでDuckDB内にテーブルとして取り込みます。

In [ ]:
# CSVからテーブルを作成
con.execute(f"""
    CREATE TABLE customers AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/customers.csv')
""")

con.execute(f"""
    CREATE TABLE products AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/products.csv')
""")

con.execute(f"""
    CREATE TABLE sales AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/sales.csv')
""")

print("3テーブルを作成しました")

# テーブル一覧の確認
tables = con.execute("SHOW TABLES").fetchall()
print("=== テーブル一覧 ===")
for t in tables:
    print(t[0])

In [ ]:
# テーブルのスキーマ（カラム情報）確認
schema = con.execute("DESCRIBE sales").fetchall()
print("=== sales テーブルのスキーマ ===")
for col in schema:
    print(f"  {col[0]:20s} {col[1]}")

In [ ]:
# 基本SELECT
rows = con.execute("""
    SELECT *
    FROM sales
    WHERE amount > 100000
    ORDER BY amount DESC
    LIMIT 10
""").fetchall()

print("=== 10万円超の売上（上位10件） ===")
for row in rows:
    print(row)

---
## 5. 集計クエリ

DuckDB は分析クエリが得意です。

In [ ]:
# 月別売上集計
rows = con.execute("""
    SELECT
        strftime(sale_date, '%Y-%m') AS year_month,
        COUNT(*)                     AS 件数,
        SUM(amount)                  AS 売上合計,
        AVG(amount)                  AS 平均単価
    FROM sales
    GROUP BY year_month
    ORDER BY year_month
""").fetchall()

print(f"{'年月':<10} {'件数':>5} {'売上合計':>12} {'平均単価':>12}")
print("-" * 45)
for row in rows:
    print(f"{row[0]:<10} {row[1]:>5} {row[2]:>12,.0f} {row[3]:>12,.0f}")

In [ ]:
# カテゴリ別売上（productsテーブルと組み合わせ）
rows = con.execute("""
    SELECT
        p.category,
        COUNT(*)       AS 件数,
        SUM(s.amount)  AS 売上合計
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    GROUP BY p.category
    ORDER BY 売上合計 DESC
""").fetchall()

print(f"{'カテゴリ':<15} {'件数':>5} {'売上合計':>12}")
print("-" * 35)
for row in rows:
    print(f"{row[0]:<15} {row[1]:>5} {row[2]:>12,.0f}")

---
## 6. JOIN クエリ

3テーブルを結合して、人間が読みやすいレポートを作ります。

In [ ]:
# 売上明細（顧客名・商品名付き）
rows = con.execute("""
    SELECT
        s.sale_id,
        s.sale_date,
        c.name        AS 顧客名,
        c.region      AS 地域,
        p.product_name AS 商品名,
        p.category    AS カテゴリ,
        s.quantity    AS 数量,
        s.amount      AS 金額
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    JOIN products  p ON s.product_id  = p.product_id
    ORDER BY s.sale_date
    LIMIT 10
""").fetchall()

print(f"{'ID':>4} {'日付':<12} {'顧客名':<12} {'商品名':<15} {'数量':>4} {'金額':>10}")
print("-" * 65)
for row in rows:
    print(f"{row[0]:>4} {str(row[1]):<12} {row[2]:<12} {row[4]:<15} {row[6]:>4} {row[7]:>10,.0f}")

In [ ]:
# 顧客別売上ランキング
rows = con.execute("""
    SELECT
        c.name       AS 顧客名,
        c.region     AS 地域,
        COUNT(*)     AS 購入回数,
        SUM(s.amount) AS 累計購入額
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name, c.region
    ORDER BY 累計購入額 DESC
""").fetchall()

print(f"{'顧客名':<12} {'地域':<8} {'購入回数':>6} {'累計購入額':>12}")
print("-" * 45)
for row in rows:
    print(f"{row[0]:<12} {row[1]:<8} {row[2]:>6} {row[3]:>12,.0f}")

## まとめ

| 機能 | 説明 |
|------|------|
| `read_csv_auto()` | CSVをそのままSQLでクエリ（インポート不要） |
| `duckdb.connect()` | インメモリDB（一時的） |
| `duckdb.connect('path.duckdb')` | ファイルDB（永続化） |
| `CREATE TABLE AS SELECT` | CSVからテーブルを作成 |
| `DESCRIBE テーブル名` | スキーマ確認 |
| `SHOW TABLES` | テーブル一覧 |
| `.fetchall()` | 結果をタプルのリストで取得（pandas不要） |